# ID5059: Group Project

- The business outcome is to **reduce passenger waiting times** while **avoiding an excess of idle drivers**. You must present something *actionable* to the Operations Manager

- Your primary task is to build predictive models for **ride demand** for the Manhattan area. These
predictions should support recommendations **about driver deployment over time and space**.
You should also model **fare-related quantities** (such as average fare or total revenue per hour)
in order to explore which locations and times are likely to be most profitable for drivers.

- For example, you could predict a typical week, or a typical
week in each season (if you see a large fluctuation between summer and winter for example).

## Preamble

In [1]:
import pandas as pd 
import numpy as np
import datetime as dt
import warnings

## Data cleaning

In [2]:
# Load in Raw data - Monthly data for 2024 (memory-efficient)
warnings.filterwarnings('ignore')

dtype_map = {
    'VendorID':              'Int8',
    'passenger_count':       'Int8',
    'trip_distance':         'float32',
    'RatecodeID':            'float32',
    'store_and_fwd_flag':    'str',
    'PULocationID':          'Int16',
    'DOLocationID':          'Int16',
    'payment_type':          'Int8',
    'fare_amount':           'float32',
    'extra':                 'float32',
    'mta_tax':               'float32',
    'tip_amount':            'float32',
    'tolls_amount':          'float32',
    'improvement_surcharge': 'float32',
    'total_amount':          'float32',
    'congestion_surcharge':  'float32',
    'Airport_fee':           'float32',
}

chunks = []
for m in range(1, 13):
    path = f'Raw Data/nyc_taxi_2024-{m:02d}.csv'
    df = pd.read_csv(path, dtype=dtype_map)
    chunks.append(df)
    print(f"Loaded {path}: {len(df):,} rows")
    del df  # free each monthly frame immediately

nyc_taxi_2024_Raw = pd.concat(chunks, ignore_index=True)
del chunks
print(f"\nTotal rows loaded: {len(nyc_taxi_2024_Raw):,}")

Loaded Raw Data/nyc_taxi_2024-01.csv: 2,964,624 rows
Loaded Raw Data/nyc_taxi_2024-02.csv: 3,007,526 rows
Loaded Raw Data/nyc_taxi_2024-03.csv: 3,582,628 rows
Loaded Raw Data/nyc_taxi_2024-04.csv: 3,514,289 rows
Loaded Raw Data/nyc_taxi_2024-05.csv: 3,723,833 rows
Loaded Raw Data/nyc_taxi_2024-06.csv: 3,539,193 rows
Loaded Raw Data/nyc_taxi_2024-07.csv: 3,076,903 rows
Loaded Raw Data/nyc_taxi_2024-08.csv: 2,979,183 rows
Loaded Raw Data/nyc_taxi_2024-09.csv: 3,633,030 rows
Loaded Raw Data/nyc_taxi_2024-10.csv: 3,833,771 rows
Loaded Raw Data/nyc_taxi_2024-11.csv: 3,646,369 rows
Loaded Raw Data/nyc_taxi_2024-12.csv: 3,668,371 rows

Total rows loaded: 41,169,720


In [3]:
nyc_taxi_2024 = nyc_taxi_2024_Raw

## Data Dictionary : Cleaned Demand Table (`taxi_clean`)

Each row represents **one pickup zone × one hour** combination.  
Only Manhattan pickup zones are included (69 TLC zones).

---

### Identifiers / Time Index

| Column | Type | Description |
|---|---|---|
| `PULocationID` | int | TLC taxi zone ID for the pickup location. Maps to the `taxi_zone_lookup.csv` reference table. |
| `pickup_date` | str (YYYY-MM-DD) | Calendar date of the pickup hour. |
| `pickup_hour` | int (0–23) | Hour of day (24-hour clock) for the pickup window. |

---

### Target Variables

| Column | Type | Description |
|---|---|---|
| `trip_count` | int | **Primary target.** Total number of trips that started in this zone during this hour. Measures ride demand. |
| `total_revenue` | float | **Secondary target.** Sum of `total_amount` (USD) across all trips in this zone-hour. Measures revenue potential. |

---

### Descriptive Summaries *(EDA only! do not use as model features)*

| Column | Type | Description |
|---|---|---|
| `avg_fare` | float | Mean base fare (`fare_amount`) in USD for trips in this zone-hour. Excludes surcharges and tips. |
| `avg_duration_min` | float | Mean trip duration in minutes for trips in this zone-hour. |

---

### Temporal Features

| Column | Type | Description |
|---|---|---|
| `pickup_dow` | int (0–6) | Day of the week: 0 = Monday, 6 = Sunday. |
| `pickup_month` | int (1–12) | Calendar month of the pickup. |
| `season` | str | Season derived from month: `Winter` (Dec–Feb), `Spring` (Mar–May), `Summer` (Jun–Aug), `Autumn` (Sep–Nov). |
| `is_weekend` | int (0/1) | 1 if the pickup falls on Saturday or Sunday, 0 otherwise. |
| `is_rush_hour` | int (0/1) | 1 if the pickup hour is in the AM rush (07:00–09:00) or PM rush (16:00–19:00), 0 otherwise. |

---

### Lag & Rolling Features: Demand (`trip_count`)

| Column | Type | Description |
|---|---|---|
| `demand_lag_1h` | float | `trip_count` for the **same zone, 1 hour earlier**. Captures short-term momentum. |
| `demand_lag_24h` | float | `trip_count` for the **same zone, 24 hours earlier**. Captures same-hour-yesterday demand. |
| `demand_lag_1w` | float | `trip_count` for the **same zone, 168 hours (1 week) earlier**. Captures weekly seasonality. |
| `demand_roll_24h` | float | Rolling 24-hour **mean** of `trip_count` for the same zone, computed over the 24 hours *before* the current hour (no data leakage). |

---

### Lag & Rolling Features: Revenue (`total_revenue`)

| Column | Type | Description |
|---|---|---|
| `revenue_lag_1h` | float | `total_revenue` for the **same zone, 1 hour earlier**. |
| `revenue_lag_24h` | float | `total_revenue` for the **same zone, 24 hours earlier**. |
| `revenue_lag_1w` | float | `total_revenue` for the **same zone, 168 hours (1 week) earlier**. |
| `revenue_roll_24h` | float | Rolling 24-hour **mean** of `total_revenue` for the same zone, computed over the 24 hours before the current hour. |

---

> **Note on NaN values:** Lag and rolling columns will be `NaN` for the first hours of the dataset (insufficient history). These rows should be dropped or imputed before model training.


In [ ]:
# Load TLC zone lookup to identify Manhattan zones reproducibly.
zones = pd.read_csv('Raw Data/taxi_zone_lookup.csv')

manhattan_zones = set(zones.loc[zones['Borough'] == 'Manhattan', 'LocationID'])
print(f"Manhattan zones loaded: {len(manhattan_zones)} zones")

# Parse datetimes (stored as strings in raw CSVs)
nyc_taxi_2024['tpep_pickup_datetime']  = pd.to_datetime(nyc_taxi_2024['tpep_pickup_datetime'])
nyc_taxi_2024['tpep_dropoff_datetime'] = pd.to_datetime(nyc_taxi_2024['tpep_dropoff_datetime'])

# Fix mixed-type store_and_fwd_flag (Y/N strings with NaN in some months)
nyc_taxi_2024['store_and_fwd_flag'] = (
    nyc_taxi_2024['store_and_fwd_flag']
    .astype(str).str.upper()
    .map({'Y': True, 'N': False})
)

# congestion_surcharge and Airport_fee are absent for some vendor records, not truly missing
nyc_taxi_2024['congestion_surcharge'] = nyc_taxi_2024['congestion_surcharge'].fillna(0)
nyc_taxi_2024['Airport_fee']          = nyc_taxi_2024['Airport_fee'].fillna(0)

# Drop invalid and voided records.
# Note: payment_type=0 rows are always the same rows as null passenger_count and
# null RatecodeID (Vendor 2 records with missing metadata), so the payment_type
# filter is sufficient to remove all of them, but each condition is kept explicit.
# RatecodeID=99 is a separate invalid code (not in the TLC data dictionary) and
# is filtered independently.
n0 = len(nyc_taxi_2024)

valid = (
    (nyc_taxi_2024['fare_amount']    >  0)  &  # negative = void/correction
    (nyc_taxi_2024['total_amount']   >  0)  &
    (nyc_taxi_2024['trip_distance']  >  0)  &  # zero-distance ghost trips
    (nyc_taxi_2024['passenger_count'] > 0)  &  # NaN rows also excluded by > 0
    (nyc_taxi_2024['payment_type']   != 0)  &  # 0 = vendor metadata missing
    (~nyc_taxi_2024['RatecodeID'].isin([99, np.nan])) &  # 99 = invalid/unknown rate
    (nyc_taxi_2024['tpep_dropoff_datetime'] > nyc_taxi_2024['tpep_pickup_datetime'])
)
nyc_taxi_2024 = nyc_taxi_2024[valid]

# Compute trip duration, then drop implausible trips
nyc_taxi_2024['trip_duration_min'] = (
    (nyc_taxi_2024['tpep_dropoff_datetime'] - nyc_taxi_2024['tpep_pickup_datetime'])
    .dt.total_seconds() / 60
)

nyc_taxi_2024 = nyc_taxi_2024[
    (nyc_taxi_2024['trip_distance']    <= 100) &  # >100 miles implausible in NYC
    (nyc_taxi_2024['fare_amount']      <= 100) &  # extreme fare outliers
    (nyc_taxi_2024['trip_duration_min'].between(1, 300))  # <1 min or >5 hrs
]

print(f"Dropped {n0 - len(nyc_taxi_2024):,} invalid/outlier rows "
        f"({(n0 - len(nyc_taxi_2024)) / n0:.1%} of total)")

# Filter to Manhattan pickups only
n1 = len(nyc_taxi_2024)
nyc_taxi_2024 = nyc_taxi_2024[nyc_taxi_2024['PULocationID'].isin(manhattan_zones)]
print(f"Dropped {n1 - len(nyc_taxi_2024):,} non-Manhattan rows "
        f"({(n1 - len(nyc_taxi_2024)) / n1:.1%} of post-cleaning total)")

# Temporal features at trip level
nyc_taxi_2024['pickup_hour']  = nyc_taxi_2024['tpep_pickup_datetime'].dt.hour
nyc_taxi_2024['pickup_dow']   = nyc_taxi_2024['tpep_pickup_datetime'].dt.dayofweek  # 0=Mon
nyc_taxi_2024['pickup_month'] = nyc_taxi_2024['tpep_pickup_datetime'].dt.month
nyc_taxi_2024['pickup_date']  = nyc_taxi_2024['tpep_pickup_datetime'].dt.normalize()
nyc_taxi_2024['season']       = nyc_taxi_2024['pickup_month'].map(
    {12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring', 4: 'Spring', 5: 'Spring',
        6: 'Summer', 7: 'Summer', 8: 'Summer',
        9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}
)
nyc_taxi_2024['is_weekend']   = nyc_taxi_2024['pickup_dow'].isin([5, 6])
nyc_taxi_2024['is_rush_hour'] = nyc_taxi_2024['pickup_hour'].isin([7, 8, 9, 16, 17, 18, 19])

# Aggregate to hourly zone-level demand table.
# trip_count and total_revenue are the two model targets.
# avg_fare and avg_duration are kept as descriptive summaries for EDA only.
# Trip-level stats unknowable at forecast time (e.g. fare_per_mile, tip_rate)
# are excluded as they cannot be used as inputs when predicting future demand.
demand = (
    nyc_taxi_2024
    .groupby(['PULocationID', 'pickup_date', 'pickup_hour'])
    .agg(
        trip_count       = ('VendorID',         'count'),  # TARGET: demand
        total_revenue    = ('total_amount',      'sum'),   # TARGET: revenue
        avg_fare         = ('fare_amount',       'mean'),  # EDA only
        avg_duration_min = ('trip_duration_min', 'mean'),  # EDA only
    )
    .reset_index()
)

# Re-attach time features (all deterministic from date + hour, so no groupby ambiguity)
demand['pickup_dow']   = demand['pickup_date'].dt.dayofweek
demand['pickup_month'] = demand['pickup_date'].dt.month
demand['season']       = demand['pickup_month'].map(
    {12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring', 4: 'Spring', 5: 'Spring',
        6: 'Summer', 7: 'Summer', 8: 'Summer',
        9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}
)
demand['is_weekend']   = demand['pickup_dow'].isin([5, 6])
demand['is_rush_hour'] = demand['pickup_hour'].isin([7, 8, 9, 16, 17, 18, 19])

demand = demand.sort_values(['PULocationID', 'pickup_date', 'pickup_hour']).reset_index(drop=True)

# Lag and rolling features for both targets.
# A pivot (hour × zone) is reindexed to a gapless hourly DatetimeIndex before
# shifting, so each shift() step is an exact time interval regardless of missing
# zone-hours in the aggregation. Zero-trip hours are filled with 0 in the pivot
# but no extra rows are added to demand. Only the original aggregated rows are kept.
demand['pickup_datetime'] = (
    demand['pickup_date'] + pd.to_timedelta(demand['pickup_hour'], unit='h')
)
full_dt_idx = pd.date_range(
    start=demand['pickup_datetime'].min(),
    end=demand['pickup_datetime'].max(),
    freq='h'
)
lookup_idx = pd.MultiIndex.from_arrays(
    [demand['pickup_datetime'], demand['PULocationID']]
)

for target, prefix in [('trip_count', 'demand'), ('total_revenue', 'revenue')]:
    pivot = (
        demand.pivot_table(
            index='pickup_datetime', columns='PULocationID',
            values=target, aggfunc='sum'
        )
        .reindex(full_dt_idx, fill_value=0)
    )
    demand[f'{prefix}_lag_1h']   = pivot.shift(1).stack().reindex(lookup_idx).values
    demand[f'{prefix}_lag_24h']  = pivot.shift(24).stack().reindex(lookup_idx).values
    demand[f'{prefix}_lag_1w']   = pivot.shift(168).stack().reindex(lookup_idx).values
    demand[f'{prefix}_roll_24h'] = (
        pivot.shift(1).rolling(24, min_periods=12).mean()
        .stack().reindex(lookup_idx).values
    )

demand = demand.drop(columns='pickup_datetime')
# drop rows with any missing lag features (e.g. first 24h of data, or zones with sparse demand)
# only about 2% of rows
demand = demand.dropna(subset= [col for col in demand.columns if 'lag' in col])

# Convert to Data Wrangler-compatible types (DW cannot serialise datetime64 or bool)
taxi_clean = demand.copy()
taxi_clean['pickup_date']  = taxi_clean['pickup_date'].astype(str)
taxi_clean['is_weekend']   = taxi_clean['is_weekend'].astype(int)
taxi_clean['is_rush_hour'] = taxi_clean['is_rush_hour'].astype(int)

print(f"\nFinal demand table: {taxi_clean.shape[0]:,} rows x {taxi_clean.shape[1]} columns")
print(f"Targets: trip_count (demand), total_revenue (revenue)")
taxi_clean

Manhattan zones loaded: 69 zones
Dropped 6,116,307 invalid/outlier rows (14.9% of total)
Dropped 3,612,671 non-Manhattan rows (10.3% of post-cleaning total)

Final demand table: 489,374 rows x 20 columns
Targets: trip_count (demand), total_revenue (revenue)


,PULocationID,pickup_date,pickup_hour,trip_count,total_revenue,avg_fare,avg_duration_min,pickup_dow,pickup_month,season,is_weekend,is_rush_hour,demand_lag_1h,demand_lag_24h,demand_lag_1w,demand_roll_24h,revenue_lag_1h,revenue_lag_24h,revenue_lag_1w,revenue_roll_24h
0,4,2024-01-01,0,20,595.679993,21.270000,20.308333,0,1,Winter,0,0,<NA>,0,0,0.000000,NaN,0.000000,0.000000,0.000000
1,4,2024-01-01,1,22,675.239990,21.677273,19.065152,0,1,Winter,0,0,20,0,0,0.869565,595.679993,0.000000,0.000000,25.899130
2,4,2024-01-01,2,27,842.650024,22.444445,18.591975,0,1,Winter,0,0,22,0,0,1.826087,675.239990,0.000000,0.000000,55.257391
3,4,2024-01-01,3,17,646.650024,26.770588,17.092157,0,1,Winter,0,0,27,0,0,3.000000,842.650024,0.000000,0.000000,91.894348
4,4,2024-01-01,4,10,182.419998,11.750000,10.368333,0,1,Winter,0,0,17,0,0,3.739130,646.650024,0.000000,0.000000,120.009567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489369,263,2024-12-31,19,122,2662.709961,12.478689,10.217077,1,12,Winter,0,1,136,47,81,57.208333,3018.010010,888.580017,1634.920044,1142.912502
489370,263,2024-12-31,20,192,4077.469971,13.208855,11.112674,1,12,Winter,0,0,122,57,88,60.333333,2662.709961,1027.989990,1700.569946,1216.834583
489371,263,2024-12-31,21,195,3747.530029,11.809231,9.770427,1,12,Winter,0,0,192,66,82,65.958333,4077.469971,1439.729980,1640.680054,1343.896249
489372,263,2024-12-31,22,148,2845.399902,11.849324,9.602252,1,12,Winter,0,0,195,54,82,71.333333,3747.530029,1081.280029,1614.469971,1440.054585


In [ ]:
print(nyc_taxi_2024['fare_amount'].describe(percentiles=[0.005, 0.05, 0.25, 0.5, 0.75, 0.95, 0.995]))